# معمل التجربة: إضافة ذاكرة المحادثة وسياق الحوار للـ RAG

أكبر مشكلة في أنظمة RAG المبتدئة هي لما المستخدم يقول: 'طب إيه مميزاته؟' أو 'اشرحلي النوع التاني بالتفصيل'.
النظام العادي مبيفهمش الضمائر دي بتعود على إيه!
هنا هنتعلم إزاي نخلي الموديل يعيد صياغة السؤال معتمداً على تاريخ الشات قبل ما يعمل بحث.

### 1. تجهيز المستندات والنظام
يلا نجهز المعرفة ونموذج Groq:

In [1]:
import tempfile
import shutil
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
temp_dir = tempfile.mkdtemp()

corpus = [
    Document(page_content="التعلم العميق يعتمد على الشبكات العصبية الاصطناعية متعددة الطبقات."),
    Document(page_content="أهم أنواع الشبكات في التعلم العميق هي شبكات CNN للصور وشبكات Transformers للنصوص."),
]

vector_store = Chroma.from_documents(corpus, embeddings, persist_directory=temp_dir)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("النظام جاهز للاختبار!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

النظام جاهز للاختبار!


### 2. بناء المسترجع الذكي وسلسلة الإجابة
هنعمل برومبت ذكي يعيد صياغة الأسئلة بالرجوع لتاريخ المحادثة:

In [2]:
context_prompt = ChatPromptTemplate.from_messages([
    ("system", "أعد صياغة سؤال المستخدم ليصبح سؤالاً مستقلاً يفهم بدون تاريخ المحادثة إذا كان يحتوي ضمائر تشير للسابق."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_retriever = create_history_aware_retriever(llm, retriever, context_prompt)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "أجب باختصار بالاعتماد على السياق التالي:\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

qa_chain = create_stuff_documents_chain(llm, qa_prompt)
full_rag = create_retrieval_chain(history_retriever, qa_chain)
print("سلسلة المحادثة الذكية جاهزة!")

سلسلة المحادثة الذكية جاهزة!


### 3. تجربة حوار متعدد الجولات
يلا نسأل سؤال أول، وبعدها نسأل سؤال بيعتمد على الضمير:

In [3]:
history = []

# الجولة الأولى
q1 = "ما هو التعلم العميق؟"
res1 = full_rag.invoke({"input": q1, "chat_history": history})
print(f"المستخدم: {q1}")
print(f"الرد: {res1['answer'].strip()}\n")

history.append(HumanMessage(content=q1))
history.append(AIMessage(content=res1["answer"]))

# الجولة الثانية بسؤال فيه ضمير
q2 = "وما هي أهم أنواعه؟"
res2 = full_rag.invoke({"input": q2, "chat_history": history})
print(f"المستخدم: {q2}")
print(f"الرد: {res2['answer'].strip()}\n")

shutil.rmtree(temp_dir, ignore_errors=True)

المستخدم: ما هو التعلم العميق؟
الرد: التعلم العميق هو فرع من الذكاء الاصطناعي يعتمد على الشبكات العصبية الاصطناعية متعددة الطبقات لتعلم تمثيلات معقدة للبيانات. أهم أنواع الشبكات فيه هي شبكات CNN للصور وشبكات Transformers للنصوص.



المستخدم: وما هي أهم أنواعه؟
الرد: أهم أنواع الشبكات في التعلم العميق هي:

1. **شبكات CNN (Convolutional Neural Networks)** – تُستخدم أساساً لمعالجة الصور والبيانات ذات البُنى المكانية.  
2. **شبكات Transformers** – تُستخدم في معالجة النصوص والبيانات المتسلسلة، وتُعتمد على آلية الانتباه (attention).



### 4. جرب بنفسك!
تقدر تضيف جولة تالتة للشات وتسأل سؤال تاني وتشوف الذاكرة شغالة إزاي.